# FIFA World Cup Winner Prediction with Neural Networks

Welcome! This notebook walks you through building a machine learning model **step by step**.

We use historical international football match data to:
1. Predict who wins a match
2. Simulate the 2026 World Cup 1,000 times to estimate who is most likely to win

**Libraries used:** pandas, numpy, scikit-learn, tensorflow/keras, matplotlib

Run each cell from top to bottom (Shift+Enter).


---
## Step 1: Load and Explore All 4 CSV Files

Before we build anything, we need to **load** our data and **look at it**.

- A **CSV file** is a spreadsheet saved as text (Comma-Separated Values).
- **pandas** is a Python library for working with tables called **DataFrames** (like Excel sheets in code).
- We print **shape** (rows × columns), **column names**, and **sample rows** for each file.


In [1]:
# Import pandas and give it the short nickname "pd" (a common convention)
import pandas as pd

# Import numpy for numerical calculations; nickname "np"
import numpy as np

# Import matplotlib for plotting graphs
import matplotlib.pyplot as plt

# Import scikit-learn tools for encoding labels, scaling numbers, and measuring accuracy
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Import TensorFlow (deep learning library) and Keras (easy neural network API)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Import pickle to save Python objects to disk so we can reuse the model later
import pickle

# Import random for Monte Carlo simulation (random tournament outcomes)
import random

# Import defaultdict: a dictionary that creates a default value for missing keys
from collections import defaultdict

# --- File paths: change these if your CSV files live in another folder ---
RESULTS_PATH = "results.csv"
SHOOTOUTS_PATH = "shootouts.csv"
FORMER_NAMES_PATH = "former_names.csv"
GOALSCORERS_PATH = "goalscorers.csv"

# pd.read_csv reads a CSV file and returns a DataFrame (table)
results_df = pd.read_csv(RESULTS_PATH)
shootouts_df = pd.read_csv(SHOOTOUTS_PATH)
former_names_df = pd.read_csv(FORMER_NAMES_PATH)
goalscorers_df = pd.read_csv(GOALSCORERS_PATH)

# Print a visual separator line (string multiplied by 60 characters)
print("=" * 60)
print("RESULTS.CSV")
print("=" * 60)
# .shape tells us (number_of_rows, number_of_columns)
print("Shape (rows, columns):", results_df.shape)
# list(...) converts column names to a plain Python list for printing
print("Columns:", list(results_df.columns))
# .head(5) shows the first 5 rows of the table
print(results_df.head())

print("\n" + "=" * 60)
print("SHOOTOUTS.CSV")
print("=" * 60)
print("Shape (rows, columns):", shootouts_df.shape)
print("Columns:", list(shootouts_df.columns))
print(shootouts_df.head())

print("\n" + "=" * 60)
print("FORMER_NAMES.CSV")
print("=" * 60)
print("Shape (rows, columns):", former_names_df.shape)
print("Columns:", list(former_names_df.columns))
print(former_names_df.head())

print("\n" + "=" * 60)
print("GOALSCORERS.CSV")
print("=" * 60)
print("Shape (rows, columns):", goalscorers_df.shape)
print("Columns:", list(goalscorers_df.columns))
print(goalscorers_df.head())


RESULTS.CSV
Shape (rows, columns): (49477, 9)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  

SHOOTOUTS.CSV
Shape (rows, columns): (678, 5)
Columns: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']
         date    home_team         away_team       winner first_shooter
0  1967-08-22        In

---
## Step 2: Clean the Data

**Cleaning** means fixing problems so the model learns from correct information.

We will:
1. **Standardize team names** using `former_names.csv` (e.g., "Upper Volta" → "Burkina Faso")
2. **Convert dates** to real date objects (not plain text)
3. **Handle missing values** (empty cells)
4. **Fix the neutral column** (TRUE/FALSE text → 1/0 numbers)

We work on a **copy** of the data so the original file stays unchanged.


In [2]:
# .copy() creates an independent copy of the DataFrame
matches_df = results_df.copy()

# Build a dictionary: old_name -> current_name from former_names.csv
name_map = {}
# .iterrows() loops over each row in the table
for _, row in former_names_df.iterrows():
    # row["former"] is the old country name; row["current"] is the modern name
    name_map[row["former"]] = row["current"]

# Define a small function that replaces old names with current names
def standardize_team_name(team_name):
    # If team_name is missing (NaN), return it unchanged
    if pd.isna(team_name):
        return team_name
    # .get(key, default) returns mapped name, or original if not in dictionary
    return name_map.get(team_name, team_name)

# Apply the function to home_team and away_team columns
matches_df["home_team"] = matches_df["home_team"].apply(standardize_team_name)
matches_df["away_team"] = matches_df["away_team"].apply(standardize_team_name)

# pd.to_datetime converts the date column from text to datetime objects
matches_df["date"] = pd.to_datetime(matches_df["date"])

# Sort all matches from oldest to newest (important for Elo and win rates)
matches_df = matches_df.sort_values("date").reset_index(drop=True)

# Fill missing text columns with the word "Unknown"
text_cols = ["tournament", "city", "country"]
for col in text_cols:
    # .fillna replaces empty/missing values
    matches_df[col] = matches_df[col].fillna("Unknown")

# Fill missing scores with 0 (very rare in this dataset)
matches_df["home_score"] = matches_df["home_score"].fillna(0)
matches_df["away_score"] = matches_df["away_score"].fillna(0)

# The neutral column is text "TRUE"/"FALSE"; convert to 1/0
matches_df["is_neutral_venue"] = matches_df["neutral"].astype(str).str.upper().eq("TRUE").astype(int)

# Print how many rows we have after cleaning
print("Matches after cleaning:", len(matches_df))
print("Sample of cleaned data:")
print(matches_df[["date", "home_team", "away_team", "home_score", "away_score", "tournament", "is_neutral_venue"]].head())


Matches after cleaning: 49477
Sample of cleaned data:
        date home_team away_team  home_score  away_score tournament  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly   
3 1875-03-06   England  Scotland         2.0         2.0   Friendly   
4 1876-03-04  Scotland   England         3.0         0.0   Friendly   

   is_neutral_venue  
0                 0  
1                 0  
2                 0  
3                 0  
4                 0  


---
## Step 3: Merge Shootouts into Results (Fix Penalty Winners)

Some knockout matches end **0-0 or tied after extra time**, then go to a **penalty shootout**.

In `results.csv`, those rows still look like a **draw** (equal scores).
In `shootouts.csv`, we know who actually **won** the shootout.

We merge the two files and create a `winner` column that uses shootout data when scores are tied.


In [3]:
# Copy shootouts and standardize team names the same way
shootouts_clean = shootouts_df.copy()
shootouts_clean["date"] = pd.to_datetime(shootouts_clean["date"])
shootouts_clean["home_team"] = shootouts_clean["home_team"].apply(standardize_team_name)
shootouts_clean["away_team"] = shootouts_clean["away_team"].apply(standardize_team_name)

# Keep only columns needed for the merge; rename winner to avoid name clash
shootouts_clean = shootouts_clean[["date", "home_team", "away_team", "winner"]]
shootouts_clean = shootouts_clean.rename(columns={"winner": "shootout_winner"})

# Merge shootout info into matches on date + both teams (like a spreadsheet VLOOKUP)
matches_df = matches_df.merge(
    shootouts_clean,
    on=["date", "home_team", "away_team"],
    how="left",  # keep all matches; shootout_winner is NaN when no shootout happened
)

# np.select chooses values based on conditions (checked in order, top to bottom)
conditions = [
    matches_df["home_score"] > matches_df["away_score"],  # home scored more
    matches_df["home_score"] < matches_df["away_score"],  # away scored more
    matches_df["shootout_winner"].notna(),  # tied score but shootout recorded
]
choices = [
    matches_df["home_team"],  # winner is home team
    matches_df["away_team"],  # winner is away team
    matches_df["shootout_winner"],  # winner from penalty shootout
]
# default=None means true draws with no shootout stay as NaN
matches_df["winner"] = np.select(conditions, choices, default=None)

# Count how many draws were fixed using shootout data
shootout_fixed = (
    (matches_df["home_score"] == matches_df["away_score"]) & (matches_df["shootout_winner"].notna())
)
print("Draws resolved by shootout:", shootout_fixed.sum())

# Show examples where shootout decided the winner
print("\nExample shootout-resolved matches:")
print(
    matches_df.loc[shootout_fixed, ["date", "home_team", "away_team", "home_score", "away_score", "winner"]].head()
)


Draws resolved by shootout: 640

Example shootout-resolved matches:
           date    home_team         away_team  home_score  away_score  \
7125 1967-08-22        India            Taiwan         1.0         1.0   
8637 1971-11-14  South Korea  Vietnam Republic         1.0         1.0   
8819 1972-05-07  South Korea              Iraq         0.0         0.0   
8836 1972-05-17     Thailand       South Korea         1.0         1.0   
8841 1972-05-19     Thailand          Cambodia         2.0         2.0   

           winner  
7125       Taiwan  
8637  South Korea  
8819         Iraq  
8836  South Korea  
8841     Thailand  


---
## Step 4: Feature Engineering — Team Elo Ratings

**Feature engineering** means creating useful number columns (features) from raw data.

### What is Elo? (Simple explanation)

**Elo** is a rating system (originally for chess) that estimates how strong a team is.

- Every team starts at **1500** (average).
- **Beat a stronger team** → your Elo goes up a lot.
- **Lose to a weaker team** → your Elo goes down a lot.
- **Draw** → both teams move a little toward each other.

We process matches **in date order** and store each team's Elo **before** each match (so we don't cheat by using future information).


In [4]:
# K controls how much one match can change ratings (higher = bigger swings)
K_FACTOR = 32

# Default starting Elo for any team we have never seen before
DEFAULT_ELO = 1500.0

# defaultdict automatically gives DEFAULT_ELO when a team is new
team_elo = defaultdict(lambda: DEFAULT_ELO)

# Create empty columns; we will fill them row by row
matches_df["home_team_elo"] = np.nan
matches_df["away_team_elo"] = np.nan

# Loop through every match from oldest to newest
for idx, row in matches_df.iterrows():
    # Get home and away team names for this match
    home_team = row["home_team"]
    away_team = row["away_team"]

    # Look up current Elo ratings BEFORE this match is played
    home_elo = team_elo[home_team]
    away_elo = team_elo[away_team]

    # Save these pre-match ratings as features for this row
    matches_df.at[idx, "home_team_elo"] = home_elo
    matches_df.at[idx, "away_team_elo"] = away_elo

    # Expected score for home team (number between 0 and 1)
    expected_home = 1.0 / (1.0 + 10 ** ((away_elo - home_elo) / 400.0))
    # Expected score for away team is the complement
    expected_away = 1.0 - expected_home

    # Actual result score for Elo update: 1=win, 0.5=draw, 0=loss
    if row["home_score"] > row["away_score"]:
        actual_home = 1.0
        actual_away = 0.0
    elif row["home_score"] < row["away_score"]:
        actual_home = 0.0
        actual_away = 1.0
    else:
        actual_home = 0.5
        actual_away = 0.5

    # Update Elo: new_elo = old_elo + K * (actual - expected)
    team_elo[home_team] = home_elo + K_FACTOR * (actual_home - expected_home)
    team_elo[away_team] = away_elo + K_FACTOR * (actual_away - expected_away)

# Show sample Elo features
print("Sample Elo features:")
print(matches_df[["date", "home_team", "away_team", "home_team_elo", "away_team_elo"]].tail())


Sample Elo features:
            date home_team  away_team  home_team_elo  away_team_elo
49472 2026-06-27  Colombia   Portugal    1919.453531    1943.855747
49473 2026-06-27    Panama    England    1729.007149    1935.700393
49474 2026-06-27   Algeria    Austria    1836.451368    1848.890949
49475 2026-06-27    Jordan  Argentina    1695.101219    2037.805386
49476 2026-06-27   Croatia      Ghana    1888.678285    1588.012858


---
## Step 5: Elo Difference Feature

The model often learns better when we give it **differences**, not just raw numbers.

`elo_difference = home_team_elo - away_team_elo`

- **Positive** → home team is rated stronger
- **Negative** → away team is rated stronger


In [5]:
# Subtract away Elo from home Elo for each match row
matches_df["elo_difference"] = matches_df["home_team_elo"] - matches_df["away_team_elo"]

# Print a few examples
print(matches_df[["home_team", "away_team", "home_team_elo", "away_team_elo", "elo_difference"]].head())


  home_team away_team  home_team_elo  away_team_elo  elo_difference
0  Scotland   England    1500.000000    1500.000000        0.000000
1   England  Scotland    1500.000000    1500.000000        0.000000
2  Scotland   England    1484.000000    1516.000000      -32.000000
3   England  Scotland    1498.530498    1501.469502       -2.939003
4  Scotland   England    1501.334159    1498.665841        2.668317


---
## Step 6: Win Rate in Last 30 Matches

Another useful signal: **recent form**.

For each team, before each match, we look at their **last 30 matches** and compute:

`win_rate = wins / number_of_recent_matches`

- **1.0** = won every recent game
- **0.0** = lost every recent game
- **0.5** = no recent history (we use 0.5 as a neutral guess)

Again, we only use matches **before** the current row (no future leakage).


In [6]:
# Store match history per team as a list of results: 1=win, 0=loss, 0.5=draw
team_history = defaultdict(list)

# Empty columns for win rates
matches_df["home_team_win_rate"] = np.nan
matches_df["away_team_win_rate"] = np.nan

# Walk through matches chronologically again
for idx, row in matches_df.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]

    # Get last 30 results for each team BEFORE this match
    home_last_30 = team_history[home_team][-30:]
    away_last_30 = team_history[away_team][-30:]

    # np.mean of [1,0,1,...] gives win rate; empty list -> use 0.5
    matches_df.at[idx, "home_team_win_rate"] = np.mean(home_last_30) if home_last_30 else 0.5
    matches_df.at[idx, "away_team_win_rate"] = np.mean(away_last_30) if away_last_30 else 0.5

    # Determine outcome of THIS match to append to history for future rows
    if row["home_score"] > row["away_score"]:
        home_result = 1.0
        away_result = 0.0
    elif row["home_score"] < row["away_score"]:
        home_result = 0.0
        away_result = 1.0
    else:
        home_result = 0.5
        away_result = 0.5

    # Append results to each team's history list
    team_history[home_team].append(home_result)
    team_history[away_team].append(away_result)

print("Sample win rate features:")
print(matches_df[["date", "home_team", "away_team", "home_team_win_rate", "away_team_win_rate"]].tail())


Sample win rate features:
            date home_team  away_team  home_team_win_rate  away_team_win_rate
49472 2026-06-27  Colombia   Portugal            0.633333            0.716667
49473 2026-06-27    Panama    England            0.600000            0.733333
49474 2026-06-27   Algeria    Austria            0.816667            0.716667
49475 2026-06-27    Jordan  Argentina            0.500000            0.816667
49476 2026-06-27   Croatia      Ghana            0.666667            0.433333


---
## Step 7: Neutral Venue Feature

We already created `is_neutral_venue` during cleaning:

- **1** = neutral venue (neither team has home advantage)
- **0** = one team is at home

World Cup matches are usually played at neutral venues.


In [ ]:
# Count how many neutral vs home matches we have
print(matches_df["is_neutral_venue"].value_counts())
print("\nSample:")
print(matches_df[["home_team", "away_team", "city", "country", "is_neutral_venue"]].head())


---
## Step 8: Tournament Weight Feature

Not every match matters equally for World Cup prediction.

We assign a **weight** (importance score):

| Tournament type | Weight |
|-----------------|--------|
| FIFA World Cup (finals) | 5 |
| Qualifiers (name contains "qualification") | 3 |
| Friendly | 1 |
| Everything else | 2 |

Higher weight tells the model that some competitions are more competitive.


In [ ]:
# Define a function that maps tournament name text to a number weight
def get_tournament_weight(tournament_name):
    # If value is missing, treat as low-importance friendly-like match
    if pd.isna(tournament_name):
        return 1
    # Convert to lowercase string for easier matching
    name = str(tournament_name).lower()
    # Exact World Cup finals (not qualifiers)
    if name == "fifa world cup":
        return 5
    # Qualification matches across all confederations
    if "qualification" in name or "qualifier" in name:
        return 3
    # Friendly matches
    if "friendly" in name:
        return 1
    # Default weight for other tournaments (Copa America, Euros, etc.)
    return 2

# Apply the function to every row's tournament column
matches_df["tournament_weight"] = matches_df["tournament"].apply(get_tournament_weight)

print(matches_df["tournament_weight"].value_counts().sort_index())
print("\nExamples:")
print(matches_df[["tournament", "tournament_weight"]].drop_duplicates().head(10))


---
## Step 9: Define the Target Label

**Target** (label) = what we want the model to predict.

- **1** = home team won
- **0** = away team won
- **Draws are removed** for now (binary win/loss only)

We use the `winner` column from Step 3 (which includes penalty shootout fixes).


In [ ]:
# Keep only matches where we know the winner (drop true draws with no shootout)
model_df = matches_df[matches_df["winner"].notna()].copy()

# home_win = 1 if home team won, 0 if away team won
model_df["home_win"] = (model_df["winner"] == model_df["home_team"]).astype(int)

# Report how many rows we dropped (draws without a shootout winner)
dropped_draws = len(matches_df) - len(model_df)
print("Rows dropped (draws without shootout winner):", dropped_draws)
print("Total rows for modeling:", len(model_df))
print("Home win rate:", round(model_df["home_win"].mean(), 3))
print(model_df[["date", "home_team", "away_team", "home_score", "away_score", "home_win"]].head())


---
## Step 10: Encode Categorical Variables with LabelEncoder

Neural networks need **numbers**, not text like "Brazil" or "Friendly".

**LabelEncoder** converts each unique text value to an integer:
- "Brazil" → 5
- "Argentina" → 2
- etc.

We encode: `home_team`, `away_team`, `tournament`, `city`, `country`.

**Important:** We fit encoders on **training data only** to avoid cheating.


---
## Step 11: Scale Features with StandardScaler

**StandardScaler** transforms each numeric column so it has:
- **Mean = 0**
- **Standard deviation = 1**

This stops big numbers (like Elo ~1600) from overpowering small numbers (like 0/1 flags).

We **fit** the scaler on training data only, then **transform** both train and test.


---
## Step 12: Train/Test Split

- **Train:** all matches with date **before 2010-01-01**
- **Test:** **FIFA World Cup** matches from **2010 onward** (2010, 2014, 2018, 2022)

This tests whether the model can predict actual World Cup results it has never seen during training.


In [ ]:
# List of numeric features we engineered
numeric_features = [
    "home_team_elo",
    "away_team_elo",
    "elo_difference",
    "home_team_win_rate",
    "away_team_win_rate",
    "is_neutral_venue",
    "tournament_weight",
]

# Categorical columns to encode
categorical_features = ["home_team", "away_team", "tournament", "city", "country"]

# TRAIN: date strictly before 2010
train_df = model_df[model_df["date"] < "2010-01-01"].copy()

# TEST: FIFA World Cup matches from 2010 onwards
test_df = model_df[
    (model_df["date"] >= "2010-01-01") & (model_df["tournament"] == "FIFA World Cup")
].copy()

print("Training rows:", len(train_df))
print("Test rows (World Cup 2010+):", len(test_df))

# Dictionary to store one LabelEncoder per categorical column
label_encoders = {}

# Fit encoders on TRAIN data only, then transform train and test
for col in categorical_features:
    le = LabelEncoder()
    # Fit learns all unique categories from training column
    train_df[col + "_encoded"] = le.fit_transform(train_df[col].astype(str))
    # For test, map unknown labels to an "unknown" bucket
    test_values = test_df[col].astype(str)
    known_classes = set(le.classes_)
    test_encoded = []
    for value in test_values:
        if value in known_classes:
            test_encoded.append(le.transform([value])[0])
        else:
            test_encoded.append(len(le.classes_))
    test_df[col + "_encoded"] = test_encoded
    label_encoders[col] = le

# Final feature column names after encoding
encoded_feature_columns = numeric_features + [c + "_encoded" for c in categorical_features]

print("Encoded feature columns:", encoded_feature_columns)


In [ ]:
# Extract X (features) and y (target) for training set
X_train = train_df[encoded_feature_columns].values.astype(float)
y_train = train_df["home_win"].values.astype(float)

# Extract X and y for test set
X_test = test_df[encoded_feature_columns].values.astype(float)
y_test = test_df["home_win"].values.astype(float)

# Create StandardScaler and fit ONLY on training features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# Transform test with training statistics (never fit on test!)
X_test_scaled = scaler.transform(X_test)

print("Feature matrix shape (train):", X_train_scaled.shape)
print("Feature matrix shape (test):", X_test_scaled.shape)


---
## Steps 13–20: Build and Compile the Keras Neural Network

A **neural network** is a stack of layers that learns patterns from data.

Our architecture (exactly as requested):

1. **Input layer** — one input per feature
2. **Dense(64, relu)** — 64 neurons, ReLU activation
3. **Dropout(0.3)** — randomly turns off 30% of neurons to prevent overfitting
4. **Dense(32, relu)**
5. **Dropout(0.2)**
6. **Dense(16, relu)**
7. **Dense(1, sigmoid)** — outputs probability between 0 and 1

**Optimizer:** Adam (efficient gradient descent)

**Loss:** binary_crossentropy (standard for yes/no problems)

**Metric:** accuracy


In [ ]:
# Number of input features equals number of columns in X_train
num_features = X_train_scaled.shape[1]

# Build a Sequential model (layers stacked one after another)
model = keras.Sequential([
    # Input layer: expects vectors of length num_features
    layers.Input(shape=(num_features,)),
    # Fully connected layer with 64 neurons and ReLU activation
    layers.Dense(64, activation="relu"),
    # Dropout: randomly drop 30% of connections during training
    layers.Dropout(0.3),
    # Second hidden layer: 32 neurons
    layers.Dense(32, activation="relu"),
    # Dropout 20%
    layers.Dropout(0.2),
    # Third hidden layer: 16 neurons
    layers.Dense(16, activation="relu"),
    # Output layer: 1 neuron with sigmoid -> probability of home win
    layers.Dense(1, activation="sigmoid"),
])

# Compile configures how the model learns
model.compile(
    optimizer="adam",  # Adam optimizer
    loss="binary_crossentropy",  # loss function for binary classification
    metrics=["accuracy"],  # track accuracy during training
)

# Print a summary table of layers and parameter counts
model.summary()


---
## Step 21: Train the Model and Plot Training Curves

We train for **100 epochs** (100 full passes through the training data).

**validation_split=0.2** means 20% of training data is held out during training to check progress.

We plot **loss** and **accuracy** for training vs validation over time.


In [ ]:
# Train the model
history = model.fit(
    X_train_scaled,  # scaled training features
    y_train,  # training labels
    epochs=100,  # number of passes through the data
    batch_size=64,  # number of samples per gradient update
    validation_split=0.2,  # use 20% of train for validation
    verbose=1,  # print progress bar each epoch
)

# Extract loss and accuracy lists from the history object
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

# Create a figure with 1 row and 2 columns of subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot training vs validation loss
axes[0].plot(train_loss, label="Train Loss")
axes[0].plot(val_loss, label="Validation Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Plot training vs validation accuracy
axes[1].plot(train_acc, label="Train Accuracy")
axes[1].plot(val_acc, label="Validation Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

# Adjust layout so plots do not overlap
plt.tight_layout()
plt.show()


---
## Step 22: Evaluate on the Test Set

We measure how well the model predicts **real World Cup matches** from 2010 onward.

- **Accuracy** — percent of correct predictions
- **Confusion matrix** — table of predicted vs actual
- **Classification report** — precision, recall, F1-score


In [ ]:
# Predict probabilities on test set (values between 0 and 1)
y_pred_prob = model.predict(X_test_scaled, verbose=0)

# Convert probabilities to hard labels: >=0.5 means predict home win (1)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Compute accuracy score
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Confusion matrix: rows=true, cols=predicted
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Detailed metrics report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Away Win (0)", "Home Win (1)"]))


---
## Step 23: predict_match(team_a, team_b) Function

This helper predicts a **single hypothetical match** using the trained model.

It uses:
- Latest Elo ratings (`team_elo` dictionary)
- Recent win rates (`team_history`)
- World Cup settings: neutral venue + tournament weight 5

It prints win probability for both teams.


In [ ]:
# Save final Elo and history state after processing all matches (for predictions)
final_team_elo = dict(team_elo)
final_team_history = {k: list(v) for k, v in team_history.items()}


def get_win_rate(team_name, history_dict):
    # Take last 30 results for a team; default 0.5 if no history
    last_30 = history_dict.get(team_name, [])[-30:]
    return float(np.mean(last_30)) if last_30 else 0.5


def encode_category(value, col_name):
    # Use saved LabelEncoder; unseen values map to "unknown" bucket
    le = label_encoders[col_name]
    value = str(value)
    if value in le.classes_:
        return float(le.transform([value])[0])
    return float(len(le.classes_))


def _build_match_features(team_a, team_b, neutral=True, tournament="FIFA World Cup"):
    # Look up Elo ratings (default 1500 for teams never seen)
    home_elo = final_team_elo.get(team_a, DEFAULT_ELO)
    away_elo = final_team_elo.get(team_b, DEFAULT_ELO)

    # Build one row of features for team_a vs team_b
    feature_row = {
        "home_team_elo": home_elo,
        "away_team_elo": away_elo,
        "elo_difference": home_elo - away_elo,
        "home_team_win_rate": get_win_rate(team_a, final_team_history),
        "away_team_win_rate": get_win_rate(team_b, final_team_history),
        "is_neutral_venue": 1 if neutral else 0,
        "tournament_weight": get_tournament_weight(tournament),
        "home_team_encoded": encode_category(team_a, "home_team"),
        "away_team_encoded": encode_category(team_b, "away_team"),
        "tournament_encoded": encode_category(tournament, "tournament"),
        "city_encoded": encode_category("Unknown", "city"),
        "country_encoded": encode_category("Unknown", "country"),
    }

    # Convert dictionary to 2D numpy array shape (1, num_features)
    X_one = np.array([[feature_row[col] for col in encoded_feature_columns]], dtype=float)
    # Scale using the same scaler fitted on training data
    return scaler.transform(X_one)


def predict_match(team_a, team_b, neutral=True, tournament="FIFA World Cup", verbose=True):
    """Predict win probability for team_a vs team_b.

    team_a is treated as the home team in the feature vector.
    For neutral World Cup games, is_neutral_venue=1 removes home advantage in features.
    """

    # Build and scale features, then run the neural network
    X_one_scaled = _build_match_features(team_a, team_b, neutral, tournament)
    prob_a = float(model.predict(X_one_scaled, verbose=0)[0][0])
    prob_b = 1.0 - prob_a

    if verbose:
        print(f"Match: {team_a} vs {team_b}")
        print(f"  {team_a} win probability: {prob_a:.2%}")
        print(f"  {team_b} win probability: {prob_b:.2%}")

    return prob_a, prob_b


# Example predictions (verbose=True prints results)
predict_match("Brazil", "Argentina")
predict_match("France", "Germany")


---
## Step 24: Monte Carlo Simulation — 2026 World Cup (48 Teams)

**Monte Carlo simulation** = run the tournament many times with randomness.

For each simulated match:
1. Get win probability from `predict_match` logic
2. **Randomly** pick a winner based on that probability

We run **1,000 full tournaments** and count how often each team wins the final.

### 2026 format (simplified for this tutorial)
- **12 groups of 4 teams** (48 teams)
- Round-robin in each group (3 matches per team)
- **Top 2** from each group advance (24 teams)
- **8 best third-place teams** also advance → **32-team knockout**
- Single-elimination knockout until one champion

> **Note:** Update `QUALIFIED_TEAMS_2026` below if the official list changes.


In [ ]:
# 48 teams for 2026 FIFA World Cup (hosts + qualified nations)
# Names match the dataset style; edit this list if needed
QUALIFIED_TEAMS_2026 = [
    "United States", "Canada", "Mexico",  # Hosts
    "Argentina", "Brazil", "Uruguay", "Colombia", "Ecuador", "Paraguay",
    "France", "Germany", "Spain", "England", "Portugal", "Netherlands", "Belgium",
    "Croatia", "Switzerland", "Austria", "Scotland", "Norway", "Denmark", "Poland", "Serbia",
    "Japan", "South Korea", "Australia", "Saudi Arabia", "Iran", "Qatar", "Jordan", "Uzbekistan",
    "Morocco", "Senegal", "Tunisia", "Algeria", "Egypt", "Ghana", "Cameroon", "Ivory Coast",
    "Costa Rica", "Panama", "Haiti", "New Zealand", "South Africa", "Curaçao", "Wales", "Cape Verde",
]

# Safety check: we need exactly 48 teams
assert len(QUALIFIED_TEAMS_2026) == 48, "QUALIFIED_TEAMS_2026 must contain exactly 48 teams"


def simulate_one_match(team_a, team_b):
    # Get probability team_a wins without printing (verbose=False)
    prob_a, _ = predict_match(team_a, team_b, neutral=True, tournament="FIFA World Cup", verbose=False)
    # Draw a random number between 0 and 1; if below prob_a, team_a wins
    if random.random() < prob_a:
        return team_a
    return team_b


def play_group_stage(teams_in_group):
    # Dictionary to track points: 3 for win, 1 for draw, 0 for loss
    points = {team: 0 for team in teams_in_group}
    # Round-robin: each pair plays once
    for i in range(len(teams_in_group)):
        for j in range(i + 1, len(teams_in_group)):
            team_a = teams_in_group[i]
            team_b = teams_in_group[j]
            winner = simulate_one_match(team_a, team_b)
            # Winner gets 3 points; loser 0 (no draws in our simulation for simplicity)
            points[winner] += 3
    # Sort teams by points descending
    ranked = sorted(points.items(), key=lambda x: x[1], reverse=True)
    # Return full ranking list of (team, points)
    return ranked


def simulate_knockout(teams):
    # Copy list so we do not modify original
    remaining = list(teams)
    # Randomly shuffle bracket each time for variety
    random.shuffle(remaining)
    # Keep eliminating until one champion remains
    while len(remaining) > 1:
        next_round = []
        # Pair adjacent teams
        for i in range(0, len(remaining), 2):
            # If odd team left, they get a bye to next round
            if i + 1 >= len(remaining):
                next_round.append(remaining[i])
            else:
                winner = simulate_one_match(remaining[i], remaining[i + 1])
                next_round.append(winner)
        remaining = next_round
    # Return the champion
    return remaining[0]


def simulate_one_world_cup():
    # Shuffle teams and split into 12 groups of 4
    teams = QUALIFIED_TEAMS_2026.copy()
    random.shuffle(teams)
    groups = [teams[i * 4:(i + 1) * 4] for i in range(12)]

    group_winners = []
    group_runners_up = []
    third_place = []

    # Play each group
    for group in groups:
        ranking = play_group_stage(group)
        group_winners.append(ranking[0][0])
        group_runners_up.append(ranking[1][0])
        third_place.append(ranking[2][0])

    # Top 2 from each group (24 teams)
    knockout_teams = group_winners + group_runners_up

    # Add 8 best third-place teams (sort third-place by random tie-break using Elo)
    third_place_sorted = sorted(
        third_place,
        key=lambda t: final_team_elo.get(t, DEFAULT_ELO),
        reverse=True,
    )
    knockout_teams += third_place_sorted[:8]

    # Run knockout phase
    champion = simulate_knockout(knockout_teams)
    return champion


# Run 1000 full tournament simulations
NUM_SIMULATIONS = 1000
win_counts = defaultdict(int)

print(f"Running {NUM_SIMULATIONS} Monte Carlo World Cup simulations...")
for sim in range(NUM_SIMULATIONS):
    champion = simulate_one_world_cup()
    win_counts[champion] += 1

# Convert counts to probabilities
win_probs = {team: count / NUM_SIMULATIONS for team, count in win_counts.items()}
# Sort teams by probability highest first
top_teams = sorted(win_probs.items(), key=lambda x: x[1], reverse=True)

print("\nTop 10 most likely 2026 World Cup winners:")
for rank, (team, prob) in enumerate(top_teams[:10], start=1):
    print(f"{rank:2d}. {team:<20} {prob:.2%}")


---
## Save Files to Reuse the Model Later (No Retraining)

After training once, save these files to disk:

| File | What it contains |
|------|------------------|
| `fifa_wc_model.keras` | Trained neural network |
| `scaler.pkl` | StandardScaler (feature scaling rules) |
| `label_encoders.pkl` | LabelEncoders for categorical columns |
| `team_elo.pkl` | Final Elo rating per team |
| `team_history.pkl` | Match history lists for win rates |
| `model_metadata.pkl` | Feature column names and constants |

Load them in a new notebook/script to call `predict_match` instantly.


In [ ]:
# Save Keras model in modern .keras format
model.save("fifa_wc_model.keras")

# Save scaler object with pickle
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save all label encoders in one dictionary file
with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

# Save final Elo ratings
with open("team_elo.pkl", "wb") as f:
    pickle.dump(final_team_elo, f)

# Save team history for win rate calculations
with open("team_history.pkl", "wb") as f:
    pickle.dump(final_team_history, f)

# Save metadata needed to rebuild feature vectors
metadata = {
    "encoded_feature_columns": encoded_feature_columns,
    "DEFAULT_ELO": DEFAULT_ELO,
    "K_FACTOR": K_FACTOR,
}

with open("model_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Saved files:")
print("  - fifa_wc_model.keras")
print("  - scaler.pkl")
print("  - label_encoders.pkl")
print("  - team_elo.pkl")
print("  - team_history.pkl")
print("  - model_metadata.pkl")


---
## Bonus: How to Load and Predict Later

Run this in a **new notebook** after saving the files above (no retraining needed).


In [ ]:
# --- Load saved artifacts (run in a separate notebook after training once) ---
import pickle
import numpy as np
from tensorflow import keras

# Load trained model
loaded_model = keras.models.load_model("fifa_wc_model.keras")

# Load scaler
with open("scaler.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)

# Load encoders
with open("label_encoders.pkl", "rb") as f:
    loaded_encoders = pickle.load(f)

# Load Elo and history
with open("team_elo.pkl", "rb") as f:
    loaded_elo = pickle.load(f)

with open("team_history.pkl", "rb") as f:
    loaded_history = pickle.load(f)

with open("model_metadata.pkl", "rb") as f:
    loaded_metadata = pickle.load(f)

print("Model and helpers loaded successfully!")
print("Feature columns:", loaded_metadata["encoded_feature_columns"])
